# Fase 7 - Reel Instagram (30s, vertical 9:16)

Genera **solo** el reel para Instagram a partir del overlay Meta Glasses ya producido (`render_obj_id_overlay`).

- Fuente: clip Meta Glasses `video-714` (34s) -> ventana de 24s
- Formato: **1080x1920 vertical** (blurred-pad, no recorta robots)
- Cards titulo/cierre + labels + color grade. Mudo (musica se agrega aparte).

Reusa el overlay si ya existe; si no, lo regenera con el mismo codigo de la pipeline.

In [1]:
import subprocess
from pathlib import Path

ROOT = Path('/workspace/FutBotMX-UAQTeam')
OUT  = ROOT / 'notebooks/fase_7_visuales/outputs'
OUT.mkdir(parents=True, exist_ok=True)
TMP  = OUT / '_reel_tmp'
TMP.mkdir(exist_ok=True)

# ---- Config del reel (editable) ----
STEM    = 'video-714_singular_display'     # clip Meta Glasses fuente
RAW     = Path('/workspace/Meta_Glasses/17Abril') / f'{STEM}.mov'
TRKDIR  = ROOT / f'outputs/inference/trackers/yolo_sam3+bytetrack/{STEM}'
JSON    = TRKDIR / f'{STEM}.json'
OVERLAY = TRKDIR / f'{STEM}_obj_id.mp4'      # overlay ya generado por la pipeline

FONT = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
W, H = 1080, 1920          # vertical 9:16 Instagram
START, CONTENT, CARD = 5.0, 24.0, 3.0   # 3 + 24 + 3 = 30s
FINAL = OUT / 'reel_instagram_30s.mp4'
print('overlay fuente:', OVERLAY, '| existe:', OVERLAY.exists())

overlay fuente: /workspace/FutBotMX-UAQTeam/outputs/inference/trackers/yolo_sam3+bytetrack/video-714_singular_display/video-714_singular_display_obj_id.mp4 | existe: True


In [2]:
# Reusa overlay si existe; si no, lo regenera con el codigo de la pipeline
if not OVERLAY.exists():
    print('overlay no existe -> generando con render_obj_id_overlay ...')
    import sys
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from src.core.track_overlay import render_obj_id_overlay
    render_obj_id_overlay(str(JSON), video_path=str(RAW),
                          output_path=str(OVERLAY), draw_masks=True,
                          trajectory_window=30)
assert OVERLAY.exists(), 'no hay overlay'
print('OK overlay listo')

OK overlay listo


In [3]:
# Helpers ffmpeg
def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2500:])
        raise RuntimeError('ffmpeg fail')

def dtxt(text, y, size, color='white', alpha=0.6):
    t = text.replace('\\', '').replace(':', '\\:').replace("'", '\u2019')
    return (f"drawtext=fontfile={FONT}:text='{t}':fontcolor={color}:"
            f"fontsize={size}:x=(w-text_w)/2:y={y}:"
            f"box=1:boxcolor=black@{alpha}:boxborderw=16")

In [4]:
# Cards titulo y cierre
def make_card(path, lines):
    vf = f'color=c=0x0a0a0a:s={W}x{H}:d={CARD}:r=30'
    for (txt, y, sz, col) in lines:
        vf += ',' + dtxt(txt, y, sz, col)
    run(['ffmpeg', '-y', '-f', 'lavfi', '-i', vf, '-frames:v', str(int(CARD*30)),
         '-r', '30', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(path)])

card_t = TMP / 'card_title.mp4'
make_card(card_t, [
    ('FutBotMX 2026', H//2 - 180, 96, 'white'),
    ('UAQ Team', H//2 - 40, 72, '0x39d98a'),
    ('Adapting SAM 3 to Robotic Soccer', H//2 + 110, 44, '0xcccccc'),
])
card_o = TMP / 'card_out.mp4'
make_card(card_o, [
    ('YOLO  ->  SAM 3  ->  Tracking', H//2 - 110, 54, 'white'),
    ('Segmentacion + Eventos + Trayectorias', H//2, 46, '0x39d98a'),
    ('github.com/RodMed0709/FutBotMX-UAQTeam', H//2 + 150, 36, '0xaaaaaa'),
])
print('cards OK')

cards OK


In [5]:
# Contenido: overlay vertical con blurred-pad (sin recortar robots) + labels
content = TMP / 'content.mp4'
top_label = dtxt('Meta Glasses POV   -   YOLO  ->  SAM 3', 150, 46)
bot_label = dtxt('ByteTrack + mascaras + estelas', 1700, 40, '0x39d98a')
fc = (
    f'[0:v]split=2[b][f];'
    f'[b]scale=-2:{H},crop={W}:{H},boxblur=22:2,eq=brightness=-0.20:saturation=0.55[bg];'
    f'[f]scale={W}:-2,eq=contrast=1.06:saturation=1.16:brightness=0.02[fg];'
    f'[bg][fg]overlay=(W-w)/2:(H-h)/2[v0];'
    f'[v0]{top_label}[v1];'
    f'[v1]{bot_label}[outv]'
)
run(['ffmpeg', '-y', '-ss', str(START), '-t', str(CONTENT), '-i', str(OVERLAY),
     '-filter_complex', fc, '-map', '[outv]', '-r', '30',
     '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(content)])
print('contenido OK')

contenido OK


In [6]:
# Concat -> reel final
lst = TMP / 'concat.txt'
lst.write_text('\n'.join(f"file '{p}'" for p in [card_t, content, card_o]) + '\n')
run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(lst),
     '-r', '30', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(FINAL)])

dur = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                      '-of', 'csv=p=0', str(FINAL)], capture_output=True, text=True).stdout.strip()
size = FINAL.stat().st_size / 1e6
print(f'\nREEL LISTO -> {FINAL}')
print(f'duracion: {dur}s | {size:.1f}MB | {W}x{H} vertical')


REEL LISTO -> /workspace/FutBotMX-UAQTeam/notebooks/fase_7_visuales/outputs/reel_instagram_30s.mp4
duracion: 30.066667s | 13.9MB | 1080x1920 vertical
